# ZarAI — RAG Pipeline + RAGAS Evaluation (Kaggle)

Kaggle version — same pipeline as the Colab notebook, adapted for Kaggle's
paths/secrets. Retrieval corpus = train split of your ~1,000-row dataset;
generator = fine-tuned Llama-3.1-8B-Instruct QLoRA adapter; evaluation =
RAGAS (Faithfulness, Answer Relevancy, Context Precision, Context Recall).

**Before running:** add your dataset and adapter as Kaggle Datasets and
attach them to this notebook (Add Input, top right), then set the paths
in the Config cell below to match. Also add `HF_TOKEN` and `GOOGLE_API_KEY`
as Kaggle Secrets (Add-ons → Secrets) rather than typing them in plaintext.

**Column names are unknown to me** — this notebook prints `df.columns.tolist()`
first. After you see the real names, edit the `COLUMN MAP` section in the
Config cell (one place, not scattered through the notebook) and re-run.


In [36]:
# 1) Install dependencies
!pip install -q -U transformers peft bitsandbytes accelerate
!pip install -q -U sentence-transformers faiss-cpu openpyxl
!pip install -q -U google-generativeai
# NOTE: we deliberately do NOT install the `ragas` package. As of mid-2026,
# ragas's own import chain (ragas/llms/base.py) still references
# langchain_community.chat_models.vertexai, a path that langchain-community
# has removed (langchain-community itself is being sunset upstream). Every
# combination of ragas/langchain-community pinning we tried still broke on
# import, because pip's resolver keeps pulling in an incompatible
# langchain-community version regardless of the ragas pin. Instead, the next
# cells implement the same four RAGAS metrics (Faithfulness, Answer
# Relevancy, Context Precision, Context Recall) as direct Gemini API calls —
# same evaluation logic, no fragile dependency chain.


## 2) Config — paths, model names, column map

In [37]:
import os, random, zipfile
import numpy as np
import pandas as pd
import torch

# ---- Paths: Kaggle attaches datasets under /kaggle/input/<dataset-slug>/ ----
DATASET_PATH   = "/kaggle/input/datasets/mahmadghafoor/urdu-agriculture-question-asnwer/urdu- agriculture Question Asnwer.xlsx"      # <- edit to your attached dataset path
ADAPTER_PATH   = "/kaggle/input/datasets/mahmadghafoor/zarai-llama31-qlora-adapter"  # <- edit; folder OR .zip both handled below
OUTPUT_DIR     = "/kaggle/working/rag_eval_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BASE_MODEL_ID  = "meta-llama/Llama-3.1-8B-Instruct"
EMBED_MODEL_ID = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

SEED = 42
TOP_K = 3
TEST_FRACTION = 0.2  # must match the split used when fine-tuning

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ---- Your dataset is WIDE format: one row per item, parallel columns per language ----
# Actual columns: id, question_urdu, question_roman_urdu, question_english,
#                 answer_urdu, answer_roman_urdu, answer_english,
#                 category, crop, season, region, difficulty_level, source_type
ID_COL = "id"
LANGUAGES = ["urdu", "roman_urdu", "english"]
META_COLS = ["category", "crop", "season", "region", "difficulty_level", "source_type"]

# After melting to long format (next cell), every downstream cell uses these
# unified column names — question/answer/language — regardless of source schema.
COL = {
    "question": "question",
    "answer":   "answer",
    "language": "language",
    "category": "category",
    "crop":     "crop",
    "season":   "season",
    "region":   "region",
    "row_id":   "id",
}


In [38]:
# HF token + Gemini key via Kaggle Secrets (Add-ons -> Secrets in the notebook editor)
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets = UserSecretsClient()

hf_token = secrets.get_secret("HF_TOKEN")
login(token=hf_token)

os.environ["GOOGLE_API_KEY"] = secrets.get_secret("GOOGLE_API_KEY")
print("Secrets loaded.")


Secrets loaded.


## 3) Load dataset — CHECK COLUMNS FIRST, then fix the Config cell above if needed

In [39]:
df = pd.read_excel(DATASET_PATH)
print("Shape (wide, one row per item):", df.shape)
print("Columns:", df.columns.tolist())
df.head(3)


Shape (wide, one row per item): (1000, 13)
Columns: ['id', 'question_urdu', 'question_roman_urdu', 'question_english', 'answer_urdu', 'answer_roman_urdu', 'answer_english', 'category', 'crop', 'season', 'region', 'difficulty_level', 'source_type']


,id,question_urdu,question_roman_urdu,question_english,answer_urdu,answer_roman_urdu,answer_english,category,crop,season,region,difficulty_level,source_type
0,1,گندم میں پیلا زنگ کی بیماری کیسے پہچانی جائے؟,Gandum mein peela zang ki bimari kaise pehchan...,How to identify yellow rust disease in wheat?,گندم میں پیلا زنگ کی علامات: پتوں پر زرد دھاری...,Gandum mein peela zang ki alamaat: patton par ...,Yellow rust in wheat shows yellow stripes on l...,disease_management,wheat,rabi,Punjab,intermediate,expert_knowledge
1,2,گندم کو پانی کب دیں؟,Gandum ko paani kab dein?,When to irrigate wheat?,گندم کو 4 سنچائی چاہیں: کراؤن روٹ (21 دن)، کشت...,Gandum ko 4 senchai chahiyein: crown root (21 ...,Wheat needs 4 irrigations: crown root (21 days...,irrigation_management,wheat,rabi,Punjab,beginner,agricultural_extension
2,3,گندم کی بہترین اقسام کون سی ہیں؟,Gandum ki behtareen aqsaam konsi hain?,What are the best wheat varieties?,پنجاب میں NARC-2009، فیصل آباد-2008، اور پاکست...,Punjab mein NARC-2009; Faisalabad-2008; aur Pa...,In Punjab; NARC-2009; Faisalabad-2008; and Pak...,variety_selection,wheat,rabi,Punjab,beginner,agricultural_extension


In [40]:
# Sanity-check the wide schema has what we expect before melting.
expected = [ID_COL] + [f"question_{l}" for l in LANGUAGES] + [f"answer_{l}" for l in LANGUAGES] + META_COLS
missing = [c for c in expected if c not in df.columns]
if missing:
    raise ValueError(
        f"Expected wide-format columns not found: {missing}\n"
        f"Actual columns are: {df.columns.tolist()}\n"
        f"Edit ID_COL / LANGUAGES / META_COLS in the Config cell to match, then re-run from there."
    )
print("Wide schema OK.")


Wide schema OK.


In [41]:
# Melt wide (id, question_<lang>, answer_<lang>, ...) into long format:
# one row per (item, language) with unified column names used by every cell below.
def melt_to_long(wide_df):
    frames = []
    for lang in LANGUAGES:
        cols = [ID_COL, f"question_{lang}", f"answer_{lang}"] + META_COLS
        sub = wide_df[cols].copy()
        sub = sub.rename(columns={f"question_{lang}": "question", f"answer_{lang}": "answer"})
        sub["language"] = lang
        frames.append(sub)
    long_df = pd.concat(frames, ignore_index=True)
    return long_df

long_df = melt_to_long(df)
print("Long shape:", long_df.shape, f"(should be {len(df)} x {len(LANGUAGES)} = {len(df) * len(LANGUAGES)})")
long_df.head(6)


Long shape: (3000, 10) (should be 1000 x 3 = 3000)


,id,question,answer,category,crop,season,region,difficulty_level,source_type,language
0,1,گندم میں پیلا زنگ کی بیماری کیسے پہچانی جائے؟,گندم میں پیلا زنگ کی علامات: پتوں پر زرد دھاری...,disease_management,wheat,rabi,Punjab,intermediate,expert_knowledge,urdu
1,2,گندم کو پانی کب دیں؟,گندم کو 4 سنچائی چاہیں: کراؤن روٹ (21 دن)، کشت...,irrigation_management,wheat,rabi,Punjab,beginner,agricultural_extension,urdu
2,3,گندم کی بہترین اقسام کون سی ہیں؟,پنجاب میں NARC-2009، فیصل آباد-2008، اور پاکست...,variety_selection,wheat,rabi,Punjab,beginner,agricultural_extension,urdu
3,4,گندم میں کارناٹ بیماری کیا ہے؟,کارناٹ بنٹ ایک فنگل بیماری ہے جو دانوں کو متاث...,disease_management,wheat,rabi,Punjab/Sindh,advanced,expert_knowledge,urdu
4,5,گندم کی بوائی کا وقت کیا ہے؟,گندم کی بوائی اکتوبر کے آخر سے نومبر کے وسط تک...,planting_calendar,wheat,rabi,All Pakistan,beginner,agricultural_extension,urdu
5,6,گندم میں یوریا کا صحیح استعمال؟,یوریا کی 3 قسطیں دیں: پہلی سنچائی پر آدھی، دوس...,fertilizer_management,wheat,rabi,Punjab,beginner,agricultural_extension,urdu


## 4) Rebuild the SAME train/test split used for fine-tuning

**Split at the ITEM level (`id`), before language expansion — not on `long_df` directly.**
Because English/Roman Urdu/Urdu versions of the same item carry the same underlying
answer, splitting `long_df` row-by-row could put e.g. the English version of an item
in train and the Urdu version of the *same* item in test. The Urdu test question would
then be answerable by retrieving its own English twin from the training corpus — a
cross-language leak that's easy to miss since the text itself doesn't match.
Splitting by `id` first guarantees every language version of a given item stays on the
same side of the split.

**Strict mode:** this raises an error instead of silently re-deriving a split if no saved
split file is found — silent re-derivation is exactly how train/test bleed happens across
notebooks without anyone noticing.

In [42]:
SPLIT_TRAIN_IDS = "/kaggle/input/zarai-split/train_item_ids.csv"   # <- edit to your attached split dataset
SPLIT_TEST_IDS  = "/kaggle/input/zarai-split/test_item_ids.csv"

FORCE_REDERIVE_SPLIT = True  # only flip this to True if you are deliberately creating the canonical split for the first time

if os.path.exists(SPLIT_TRAIN_IDS) and os.path.exists(SPLIT_TEST_IDS) and not FORCE_REDERIVE_SPLIT:
    train_ids = pd.read_csv(SPLIT_TRAIN_IDS)["id"].tolist()
    test_ids  = pd.read_csv(SPLIT_TEST_IDS)["id"].tolist()
    print(f"Loaded canonical item-level split: {len(train_ids)} train items / {len(test_ids)} test items.")
elif FORCE_REDERIVE_SPLIT:
    from sklearn.model_selection import train_test_split

    unique_items = df[[ID_COL, "category"]].drop_duplicates(subset=ID_COL)
    train_ids, test_ids = train_test_split(
        unique_items[ID_COL],
        test_size=TEST_FRACTION,
        random_state=SEED,
        stratify=unique_items["category"].astype(str),
    )
    train_ids, test_ids = train_ids.tolist(), test_ids.tolist()

    os.makedirs("/kaggle/working/split", exist_ok=True)
    pd.DataFrame({"id": train_ids}).to_csv("/kaggle/working/split/train_item_ids.csv", index=False)
    pd.DataFrame({"id": test_ids}).to_csv("/kaggle/working/split/test_item_ids.csv", index=False)
    print(f"Canonical item-level split created: {len(train_ids)} train / {len(test_ids)} test.")
    print("IMPORTANT: upload these two CSVs as a Kaggle Dataset (e.g. 'zarai-split') and")
    print("attach it to every future notebook (fine-tuning, SBERT eval, RAG eval) so they")
    print("all load the SAME split instead of re-deriving one.")
else:
    raise FileNotFoundError(
        "No canonical split found at the configured SPLIT_TRAIN_IDS / SPLIT_TEST_IDS paths, "
        "and FORCE_REDERIVE_SPLIT is False.\n"
        "Either attach the Kaggle Dataset containing your saved split, or set "
        "FORCE_REDERIVE_SPLIT=True to create one now (only do this once, for the whole project)."
    )

# Expand the item-level split to the long (per-language) dataframe used by every cell below.
train_df = long_df[long_df[ID_COL].isin(train_ids)].reset_index(drop=True)
test_df  = long_df[long_df[ID_COL].isin(test_ids)].reset_index(drop=True)

print(f"Train (retrieval corpus): {len(train_df)} rows across {len(LANGUAGES)} languages")
print(f"Test (held-out eval):     {len(test_df)} rows across {len(LANGUAGES)} languages")
print(test_df['language'].value_counts())


Canonical item-level split created: 800 train / 200 test.
IMPORTANT: upload these two CSVs as a Kaggle Dataset (e.g. 'zarai-split') and
attach it to every future notebook (fine-tuning, SBERT eval, RAG eval) so they
all load the SAME split instead of re-deriving one.
Train (retrieval corpus): 2400 rows across 3 languages
Test (held-out eval):     600 rows across 3 languages
language
urdu          200
roman_urdu    200
english       200
Name: count, dtype: int64


## 5) Build the retrieval corpus (train split only) and FAISS index

In [43]:
from sentence_transformers import SentenceTransformer
import faiss

embedder = SentenceTransformer(EMBED_MODEL_ID)

def build_corpus_text(row):
    return f"Q: {row[COL['question']]}\nA: {row[COL['answer']]}"

corpus_texts = train_df.apply(build_corpus_text, axis=1).tolist()
meta_cols = [COL[c] for c in ["question", "answer", "category", "crop", "season", "region", "language"]]
corpus_meta = train_df[meta_cols].to_dict("records")

print("Encoding corpus...")
corpus_embeddings = embedder.encode(
    corpus_texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True
)

dim = corpus_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(np.array(corpus_embeddings, dtype="float32"))
print(f"FAISS index built: {index.ntotal} vectors, dim={dim}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Encoding corpus...


Batches:   0%|          | 0/38 [00:00<?, ?it/s]

FAISS index built: 2400 vectors, dim=768


In [44]:
def retrieve(query, k=TOP_K):
    q_emb = embedder.encode([query], normalize_embeddings=True)
    scores, idxs = index.search(np.array(q_emb, dtype="float32"), k)
    hits = []
    for score, idx in zip(scores[0], idxs[0]):
        if idx == -1:
            continue
        hits.append({"score": float(score), "text": corpus_texts[idx], **corpus_meta[idx]})
    return hits

sample_q = test_df.iloc[0][COL["question"]]
for h in retrieve(sample_q):
    print(round(h["score"], 3), "-", str(h[COL["question"]])[:80])


0.787 - گندم میں نئی اقسام کی بوائی کا بہترین وقت؟
0.773 - گندم میں نئی اقسام کی بوائی کا وقت؟
0.741 - گندم کو پانی کب دیں؟


## 6) Load the fine-tuned Llama-3.1-8B-Instruct (QLoRA adapter)

In [45]:
def resolve_adapter_path(path):
    # Handles: a plain adapter folder, or a .zip that needs extracting first
    # (matches your 'resumable' fine-tuning notebook's save format).
    if path.endswith(".zip"):
        extract_dir = "/kaggle/working/adapter_extracted"
        os.makedirs(extract_dir, exist_ok=True)
        with zipfile.ZipFile(path, "r") as z:
            z.extractall(extract_dir)
        print(f"Extracted adapter zip -> {extract_dir}")
        return extract_dir

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"ADAPTER_PATH does not exist: {path}\n"
            f"Check the Kaggle Dataset is attached (Add Input) and the path matches "
            f"/kaggle/input/<your-dataset-slug>/<subfolder>."
        )

    contents = os.listdir(path)
    if "adapter_config.json" not in contents:
        # maybe there's exactly one subfolder or zip inside — try to find it
        candidates = [c for c in contents if c.endswith(".zip")]
        if candidates:
            return resolve_adapter_path(os.path.join(path, candidates[0]))
        subdirs = [c for c in contents if os.path.isdir(os.path.join(path, c))]
        if len(subdirs) == 1:
            return resolve_adapter_path(os.path.join(path, subdirs[0]))
        raise FileNotFoundError(
            f"No adapter_config.json found in {path}. Contents: {contents}\n"
            f"This doesn't look like a PEFT adapter folder — double check ADAPTER_PATH."
        )
    return path

ADAPTER_PATH = resolve_adapter_path(ADAPTER_PATH)
print("Resolved adapter path:", ADAPTER_PATH)
print("Contents:", os.listdir(ADAPTER_PATH))


Resolved adapter path: /kaggle/input/datasets/mahmadghafoor/zarai-llama31-qlora-adapter
Contents: ['adapter_model.safetensors', 'training_args.bin', 'adapter_config.json', 'README.md', 'tokenizer.json', 'tokenizer_config.json', 'chat_template.jinja']


In [46]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,   # bf16 throughout, per your earlier T4 findings
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
print("Fine-tuned model + adapter loaded.")


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Fine-tuned model + adapter loaded.


## 7) RAG generation function

In [47]:
RAG_SYSTEM_PROMPT = (
    "You are ZarAI, an agricultural assistant for Pakistani farmers. "
    "Answer the farmer's question using ONLY the information in the provided context. "
    "If the context does not contain the answer, say so honestly rather than guessing. "
    "Answer in the same language/script as the question."
)

def build_prompt(question, contexts):
    context_block = "\n\n".join(f"[{i+1}] {c['text']}" for i, c in enumerate(contexts))
    user_msg = f"Context:\n{context_block}\n\nQuestion: {question}"
    messages = [
        {"role": "system", "content": RAG_SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def rag_answer(question, k=TOP_K, max_new_tokens=256):
    contexts = retrieve(question, k=k)
    prompt = build_prompt(question, contexts)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=1.0,
        pad_token_id=tokenizer.pad_token_id,
    )
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return text.strip(), [c["text"] for c in contexts]

ans, ctx = rag_answer(sample_q)
print("Q:", sample_q)
print("A:", ans)
print("Contexts used:", len(ctx))


Q: گندم کی بوائی کا وقت کیا ہے؟
A: گندم کی نئی اقسام کی بوائی کا بہترین وقت اکتوبر کے آخر سے نومبر کے پہلے ہفتے تک ہے۔
Contexts used: 3


## 8) Run the RAG pipeline over the held-out test set

In [48]:
import shutil

# If you already ran generation once and saved the output as a Kaggle Dataset,
# attach it (Add Input) and copy it into place here so the next cell's
# checkpoint check finds it and skips the ~80+ minute regeneration.
# Set to None to skip this step (e.g. first-ever run, nothing to copy yet).
UPLOADED_CHECKPOINT_PATH = "/kaggle/input/datasets/mahmadghafoor/rag-generation-results/rag_generation_results.csv"  # <- edit to your real attached path, or set to None

os.makedirs(OUTPUT_DIR, exist_ok=True)
target_path = os.path.join(OUTPUT_DIR, "rag_generation_results.csv")

if UPLOADED_CHECKPOINT_PATH and os.path.exists(UPLOADED_CHECKPOINT_PATH):
    shutil.copy(UPLOADED_CHECKPOINT_PATH, target_path)
    print(f"Copied uploaded checkpoint -> {target_path}")
elif UPLOADED_CHECKPOINT_PATH:
    print(f"UPLOADED_CHECKPOINT_PATH is set but not found at that path — check the "
          f"attached dataset's real mounted path in the sidebar and fix the path above.")
else:
    print("No uploaded checkpoint configured — the next cell will run generation from scratch.")


Copied uploaded checkpoint -> /kaggle/working/rag_eval_outputs/rag_generation_results.csv


In [49]:
import ast
from tqdm.auto import tqdm

checkpoint_path = os.path.join(OUTPUT_DIR, "rag_generation_results.csv")

# Skip the (slow — ~80+ min for the full held-out set) generation loop if a
# checkpoint already exists on disk from a previous run. Delete the file (or
# set FORCE_REGENERATE=True) if you want to force a fresh run instead.
FORCE_REGENERATE = False

if os.path.exists(checkpoint_path) and not FORCE_REGENERATE:
    rag_results_df = pd.read_csv(checkpoint_path)
    # retrieved_contexts was saved as a stringified list — parse it back to a real list
    rag_results_df["retrieved_contexts"] = rag_results_df["retrieved_contexts"].apply(ast.literal_eval)
    records = rag_results_df.to_dict("records")
    print(f"Loaded existing checkpoint -> {checkpoint_path} ({len(records)} rows). Skipped regeneration.")
else:
    records = []
    for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
        answer, contexts = rag_answer(row[COL["question"]])
        records.append({
            "question": row[COL["question"]],
            "reference_answer": row[COL["answer"]],
            "rag_answer": answer,
            "retrieved_contexts": contexts,
            "language": row.get(COL["language"], None),
            "category": row.get(COL["category"], None),
            "crop": row.get(COL["crop"], None),
        })

    rag_results_df = pd.DataFrame(records)
    rag_results_df.to_csv(checkpoint_path, index=False)
    print(f"Saved intermediate RAG outputs -> {checkpoint_path}")

rag_results_df.head()


Loaded existing checkpoint -> /kaggle/working/rag_eval_outputs/rag_generation_results.csv (600 rows). Skipped regeneration.


,question,reference_answer,rag_answer,retrieved_contexts,language,category,crop
0,گندم کی بوائی کا وقت کیا ہے؟,گندم کی بوائی اکتوبر کے آخر سے نومبر کے وسط تک...,گندم کی نئی اقسام کی بوائی کا بہترین وقت اکتوب...,[Q: گندم میں نئی اقسام کی بوائی کا بہترین وقت؟...,urdu,planting_calendar,wheat
1,گندم کی کٹائی کا صحیح وقت کیا ہے؟,جب دانہ سخت ہو جائے اور پودا سنہری ہو جائے تو ...,گندم کی کٹائی کا صحیح وقت نہیں بتایا گیا ہے۔,[Q: Best sowing time for new wheat varieties?\...,urdu,post_harvest,wheat
2,چاول میں جھلسا بیماری کا علاج؟,ٹرائی سائیکلازول یا آئیسوپروتھیولین اسپرے کریں...,میں نے چاول میں جھلسا بیماری کا علاج کے بارے م...,[Q: rice میں سڑن کی بیماری کا علاج کیا ہے؟\nA:...,urdu,disease_management,rice
3,گنے میں سرخ سڑاند کا علاج؟,صرف صحت مند بیج استعمال کریں۔ بیج کو کاربینڈاز...,گنے میں سرخ سڑاند کا علاج تنے کے اندر سرخ رنگ ...,[Q: کپاس میں گلابی سنڈی کا علاج؟\nA: فیرومون ٹ...,urdu,disease_management,sugarcane
4,ٹماٹر میں جھلسا بیماری کا علاج؟,منکوزیب یا کلوروتھالونل اسپرے کریں۔ باری باری ...,میں نے یہ معلومات نہیں ملی ہیں کہ ٹماٹر میں جھ...,[Q: tomato میں لفحہ کی بیماری کا علاج کیا ہے؟\...,urdu,disease_management,tomato


## 9) RAGAS-style evaluation — implemented directly against Gemini

Faithfulness, Answer Relevancy, Context Precision, and Context Recall, computed
with direct Gemini API calls instead of the `ragas` package (see install-cell
note above for why). Each metric prompts Gemini for a structured JSON judgment,
matching the same definitions RAGAS itself uses:

- **Faithfulness** — split the answer into claims, check each against the retrieved context, score = fraction supported.
- **Answer Relevancy** — how directly the answer addresses the question (0–1).
- **Context Precision** — for each retrieved chunk, is it relevant; scored rank-aware (precision@k averaged over relevant hits), so relevant chunks ranked earlier score higher.
- **Context Recall** — split the reference answer into statements, check how many are attributable to the retrieved context.

In [50]:
import google.generativeai as genai
import json, re, time

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

# This account is a "new user" and is fully blocked from every 2.x-generation
# model (2.5-flash, 2.5-flash-lite, 2.0-flash all 404 as "no longer available
# to new users"). Only 3.x-generation models are reachable at all. Quotas on
# these vary a lot and can be VERY low (gemini-3.6-flash: 20 requests/DAY,
# not per minute) — so rather than pick one model and hope, we keep a POOL
# of working models and rotate to the next one automatically the moment a
# DAILY quota (not just RPM) is hit on the current one.
CANDIDATE_JUDGE_MODELS = [
    "gemini-3.5-flash-lite",
    "gemini-3.6-flash",
    "gemini-3.6-flash-lite",
    "gemini-flash-latest",
]

def build_model_pool(candidates):
    """Test-call every candidate once and keep all that actually work,
    not just the first — we need a rotation pool, not a single pick."""
    pool = []
    print("Testing candidate judge models:")
    for name in candidates:
        try:
            m = genai.GenerativeModel(name)
            resp = m.generate_content("Reply with exactly: ok")
            _ = resp.text
            print(f"  [OK] {name}")
            pool.append((name, m))
        except Exception as e:
            print(f"  [FAILED] {name}: {str(e)[:150]}")
    if not pool:
        raise RuntimeError(
            "No candidate judge models are reachable on this API key at all. "
            "Check https://ai.google.dev/gemini-api/docs/models for current "
            "model names available to your account and update CANDIDATE_JUDGE_MODELS."
        )
    return pool

MODEL_POOL = build_model_pool(CANDIDATE_JUDGE_MODELS)
print(f"Working model pool ({len(MODEL_POOL)}): {[n for n, _ in MODEL_POOL]}")
active_model_idx = 0  # index into MODEL_POOL of the model currently in use

def is_daily_quota_error(e):
    s = str(e)
    return "429" in s and ("PerDay" in s or "RequestsPerDay" in s or "daily" in s.lower())

def call_judge_json(prompt, max_retries=4, backoff=6):
    """Calls the currently active pool model. On a DAILY quota error,
    rotates to the next pool model immediately (retrying the same model
    is pointless — daily quotas don't clear for ~24h). On an RPM (non-daily)
    429, backs off and retries the same model instead."""
    global active_model_idx
    last_err = None
    attempts_this_call = 0
    while attempts_this_call < max_retries * len(MODEL_POOL):
        name, model = MODEL_POOL[active_model_idx]
        try:
            resp = model.generate_content(prompt)
            text = resp.text.strip()
            text = re.sub(r"^```(json)?\s*|\s*```$", "", text.strip())
            return json.loads(text)
        except Exception as e:
            last_err = e
            attempts_this_call += 1
            if is_daily_quota_error(e):
                print(f"  [daily quota hit on {name}] rotating to next model in pool...")
                active_model_idx = (active_model_idx + 1) % len(MODEL_POOL)
                if active_model_idx == 0:
                    # cycled through every model in the pool with no luck
                    raise RuntimeError(
                        "All models in the pool have hit their daily quota. "
                        "Stopping here — re-run tomorrow to continue from the "
                        "checkpoint (already-scored rows will be skipped)."
                    ) from e
                continue
            else:
                wait = backoff * ((attempts_this_call % max_retries) + 1)
                time.sleep(wait)
    raise RuntimeError(f"Judge call failed after exhausting retries/pool: {last_err}")


def score_all_metrics(question, answer, contexts, reference_answer):
    """Single combined call covering all 4 RAGAS-style metrics at once —
    cuts total API calls 4x versus one-call-per-metric."""
    ctx_block = "\n".join(f"[{i+1}] {c}" for i, c in enumerate(contexts))
    prompt = f"""You are evaluating a RAG (retrieval-augmented generation) system's output.
Compute FOUR scores based on the question, retrieved contexts, generated answer, and reference answer.

Question: {question}

Retrieved Contexts:
{ctx_block}

Generated Answer: {answer}

Reference Answer: {reference_answer}

1. FAITHFULNESS: Break the Generated Answer into individual factual claims. For each,
   decide if it is directly supported by the Retrieved Contexts. Score = fraction supported (0.0-1.0).
2. ANSWER RELEVANCY: Rate 0.0-1.0 how directly and completely the Generated Answer addresses the Question.
3. CONTEXT RELEVANCE: For EACH numbered context above, is it relevant for answering the Question
   (i.e. contains information needed for the Reference Answer)? One true/false per context, in order.
4. CONTEXT RECALL: Break the Reference Answer into individual factual statements. For each,
   decide if it can be attributed to (found in) the Retrieved Contexts. Score = fraction attributed (0.0-1.0).

Respond ONLY with JSON, no other text:
{{"faithfulness_score": 0.0, "answer_relevancy_score": 0.0, "context_relevance": [true, false], "context_recall_score": 0.0}}"""
    return call_judge_json(prompt)


def context_precision_from_relevance(relevance):
    """Rank-aware precision@k averaged over relevant hits — relevant chunks
    ranked earlier score higher, same as RAGAS's own definition."""
    if not relevance:
        return None
    if not any(relevance):
        return 0.0
    hits, precisions = 0, []
    for i, is_rel in enumerate(relevance):
        if is_rel:
            hits += 1
            precisions.append(hits / (i + 1))
    return sum(precisions) / hits

print("Judge function ready (multi-model rotation, combined single-call version).")


Testing candidate judge models:
  [OK] gemini-3.5-flash-lite
  [OK] gemini-3.6-flash
  [FAILED] gemini-3.6-flash-lite: 404 models/gemini-3.6-flash-lite is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the
  [FAILED] gemini-flash-latest: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/ge
Working model pool (2): ['gemini-3.5-flash-lite', 'gemini-3.6-flash']
Judge function ready (multi-model rotation, combined single-call version).


In [53]:
# Quick smoke test first: just 10 rows total (not stratified) to confirm the
# whole judge pipeline actually works end-to-end without burning much of the
# daily quota. Once this runs clean, flip SMOKE_TEST_ONLY = False to switch
# to the full stratified sample (10/language = 30 total).
SMOKE_TEST_ONLY = True
SMOKE_TEST_SIZE = 10
RAGAS_SAMPLE_PER_LANGUAGE = 10  # used only when SMOKE_TEST_ONLY = False -> 10 x 3 languages = 30 rows

records_df = pd.DataFrame(records)

if SMOKE_TEST_ONLY:
    sample_df = records_df.sample(n=SMOKE_TEST_SIZE, random_state=SEED)
    print(f"SMOKE TEST: sampled {len(sample_df)} rows total (not stratified by language).")
else:
    sample_df = records_df.groupby("language", group_keys=False).sample(
        n=RAGAS_SAMPLE_PER_LANGUAGE, random_state=SEED
    )
    print(f"FULL SAMPLE: sampled {len(sample_df)} rows ({RAGAS_SAMPLE_PER_LANGUAGE} per language).")

sampled_records = sample_df.to_dict("records")
print(sample_df["language"].value_counts())


SMOKE TEST: sampled 10 rows total (not stratified by language).
language
urdu          5
english       3
roman_urdu    2
Name: count, dtype: int64


In [54]:
from tqdm.auto import tqdm

SLEEP_BETWEEN_CALLS = 5.0
ragas_checkpoint_path = os.path.join(OUTPUT_DIR, "ragas_scores_checkpoint.csv")

# RESUMABLE: if a checkpoint from a previous (possibly quota-exhausted) run
# exists, load it and skip rows already scored — matches the same resumable
# pattern used elsewhere in this project (auto-detect saved progress, only
# do the remaining work).
if os.path.exists(ragas_checkpoint_path):
    ragas_scores_df = pd.read_csv(ragas_checkpoint_path)
    already_scored = set(ragas_scores_df["question"])
    remaining = [r for r in sampled_records if r["question"] not in already_scored]
    print(f"Resuming: {len(ragas_scores_df)} rows already scored, {len(remaining)} remaining.")
else:
    ragas_scores_df = pd.DataFrame(columns=["question", "language", "faithfulness",
                                             "answer_relevancy", "context_precision", "context_recall"])
    remaining = sampled_records
    print(f"Starting fresh: {len(remaining)} rows to score.")

new_rows = []
try:
    for r in tqdm(remaining, desc="RAGAS-style scoring (sample)"):
        result = score_all_metrics(r["question"], r["rag_answer"], r["retrieved_contexts"], r["reference_answer"])
        relevance = result.get("context_relevance", [])
        new_rows.append({
            "question": r["question"],
            "language": r["language"],
            "faithfulness": result.get("faithfulness_score"),
            "answer_relevancy": result.get("answer_relevancy_score"),
            "context_precision": context_precision_from_relevance(relevance),
            "context_recall": result.get("context_recall_score"),
        })
        time.sleep(SLEEP_BETWEEN_CALLS)
except RuntimeError as e:
    print(f"\nStopped early: {e}")
finally:
    # Save whatever completed, even on early stop, so nothing already-scored is lost.
    if new_rows:
        ragas_scores_df = pd.concat([ragas_scores_df, pd.DataFrame(new_rows)], ignore_index=True)
    ragas_scores_df.to_csv(ragas_checkpoint_path, index=False)
    print(f"Saved {len(ragas_scores_df)} total scored rows -> {ragas_checkpoint_path}")

ragas_scores_df.head()


Starting fresh: 10 rows to score.


RAGAS-style scoring (sample):   0%|          | 0/10 [00:00<?, ?it/s]

Saved 10 total scored rows -> /kaggle/working/rag_eval_outputs/ragas_scores_checkpoint.csv


/tmp/ipykernel_58/4202616716.py:40: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  ragas_scores_df = pd.concat([ragas_scores_df, pd.DataFrame(new_rows)], ignore_index=True)


,question,language,faithfulness,answer_relevancy,context_precision,context_recall
0,پیاز کی پیداوار بڑھانے کے لیے کیا کریں؟,urdu,1.0,1.0,0.583333,0.0
1,How to protect wheat from heat stress in 2026?,english,1.0,0.0,1.000000,0.6
2,How to control Cyanamide in cotton?,english,1.0,1.0,1.000000,0.0
3,لہسن میں بوائی کے بعد پہلی سنچائی کب؟,urdu,1.0,1.0,0.500000,1.0
4,vegetables میں کمپوسٹ کا استعمال کریں؟,urdu,1.0,1.0,1.000000,1.0


## 10) Aggregate results — overall and by language

In [55]:
# ragas_scores_df already carries question/language columns (added in the
# scoring loop), and is a SAMPLE (90 rows) of the full 600-row generation
# output — so we report on the sample directly rather than concatenating
# by position, which would misalign now that row counts differ.

metric_cols = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
metric_cols = [c for c in metric_cols if c in ragas_scores_df.columns]

print(f"=== RAGAS-style scores on the {len(ragas_scores_df)}-row sample ===")
print(ragas_scores_df[metric_cols].describe().loc[["mean", "50%", "std", "min", "max"]])

print("\n=== By language ===")
print(ragas_scores_df.groupby("language")[metric_cols].mean().round(3))

# Full RAG generation output (all 600 rows) saved separately from the
# smaller RAGAS-scored sample, since they're different row counts.
gen_out_path = os.path.join(OUTPUT_DIR, "rag_generation_full.xlsx")
rag_results_df.to_excel(gen_out_path, index=False)

ragas_out_path = os.path.join(OUTPUT_DIR, "ragas_scores_sample.xlsx")
ragas_scores_df.to_excel(ragas_out_path, index=False)

print(f"\nSaved full RAG generation output ({len(rag_results_df)} rows) -> {gen_out_path}")
print(f"Saved RAGAS-style scores ({len(ragas_scores_df)}-row sample) -> {ragas_out_path}")
print("(download both from the notebook's Output tab)")


=== RAGAS-style scores on the 10-row sample ===
      faithfulness  answer_relevancy  context_precision  context_recall
mean      0.900000          0.880000           0.858333        0.585000
50%       1.000000          1.000000           1.000000        0.635000
std       0.316228          0.315524           0.229230        0.381233
min       0.000000          0.000000           0.500000        0.000000
max       1.000000          1.000000           1.000000        1.000000

=== By language ===
            faithfulness  answer_relevancy  context_precision  context_recall
language                                                                     
english              1.0             0.667              1.000           0.423
roman_urdu           0.5             1.000              1.000           0.750
urdu                 1.0             0.960              0.717           0.616

Saved full RAG generation output (600 rows) -> /kaggle/working/rag_eval_outputs/rag_generation_full.xlsx
Sav

## 11) Sanity checks

- **Leakage check:** spot-check `retrieved_contexts` for a few test rows for near-duplicates of the question itself.
- **Low scorers:** filter `full_df` for `faithfulness < 0.4` and read manually.
- **Judge consistency:** Gemini may be weaker at judging Urdu-script/Roman Urdu than English — eyeball a sample per language.
- **Compare against your non-RAG fine-tuned baseline (80.6 mean SBERT)** to see whether RAG helps, hurts, or is noise —
  and remember: since the RAG corpus overlaps with the fine-tuning data, this comparison can't fully isolate retrieval's
  contribution from memorization. See the note on the missing base-model+RAG ablation from our last discussion.
